# ASL Alphabet MediaPipe Pipeline



This notebook downloads the ASL Alphabet dataset, extracts normalized MediaPipe hand landmarks, saves `landmarks.npy`, creates train/val/test splits, defines reusable inference helpers, and captures a few webcam test frames.



Use it locally in VS Code/Jupyter for the webcam section. Configure Kaggle credentials before running the dataset download cell.


In [4]:
import subprocess
import sys

required_packages = [
    "kagglehub",
    "mediapipe",
    "opencv-python",
    "scikit-learn",
    "numpy",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required_packages])
print("Installed:", ", ".join(required_packages))

Installed: kagglehub, mediapipe, opencv-python, scikit-learn, numpy


In [9]:
from collections import Counter

from pathlib import Path

from urllib.request import urlretrieve

import csv

import json



import cv2

import mediapipe as mp

import numpy as np

from mediapipe.tasks.python import BaseOptions

from mediapipe.tasks.python.vision import HandLandmarker

from mediapipe.tasks.python.vision import HandLandmarkerOptions

from mediapipe.tasks.python.vision import RunningMode

from sklearn.model_selection import train_test_split



PROJECT_ROOT = Path.cwd()

KAGGLE_DATASET = "grassknoted/asl-alphabet"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

HAND_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

HAND_LANDMARKER_MODEL_PATH = PROJECT_ROOT / "hand_landmarker.task"



RANDOM_SEED = 42

MAX_SAMPLES_PER_CLASS = 50

CAMERA_INDEX = 0

WEBCAM_FRAME_COUNT = 6



LANDMARKS_PATH = PROJECT_ROOT / "landmarks.npy"

LABELS_PATH = PROJECT_ROOT / "labels.npy"

METADATA_PATH = PROJECT_ROOT / "landmark_metadata.csv"

FAILED_IMAGES_PATH = PROJECT_ROOT / "failed_images.txt"

SPLITS_PATH = PROJECT_ROOT / "splits.npz"

LABEL_MAP_PATH = PROJECT_ROOT / "label_to_index.json"

WEBCAM_OUTPUT_DIR = PROJECT_ROOT / "webcam_frames"

WEBCAM_LANDMARKS_PATH = PROJECT_ROOT / "webcam_landmarks.npy"



def ensure_hand_landmarker_model(

    model_path=HAND_LANDMARKER_MODEL_PATH,

    model_url=HAND_LANDMARKER_MODEL_URL,

):

    model_path = Path(model_path)

    if model_path.exists():

        return model_path



    model_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Downloading hand landmarker model to {model_path}...")

    urlretrieve(model_url, model_path)

    return model_path





print(PROJECT_ROOT)


c:\Users\samri\OneDrive\Desktop\sign-language\Sign-Language-Project


## Dataset Download



The next cell downloads `grassknoted/asl-alphabet` from Kaggle. Make sure your Kaggle credentials are configured before you run it.


In [6]:
def download_asl_alphabet_dataset(dataset_slug=KAGGLE_DATASET):

    import kagglehub



    return Path(kagglehub.dataset_download(dataset_slug)).resolve()





def locate_class_root(dataset_dir):

    candidates = []



    def inspect_directory(path):

        if not path.is_dir():

            return

        child_dirs = [child for child in path.iterdir() if child.is_dir()]

        if len(child_dirs) < 20:

            return



        class_like_count = 0

        for child_dir in child_dirs:

            if any(

                item.is_file() and item.suffix.lower() in IMAGE_EXTENSIONS

                for item in child_dir.iterdir()

            ):

                class_like_count += 1



        if class_like_count >= 20:

            candidates.append((class_like_count, path))



    inspect_directory(dataset_dir)

    for path in dataset_dir.rglob("*"):

        inspect_directory(path)



    if not candidates:

        raise FileNotFoundError(

            f"Could not find a class-folder root under {dataset_dir}. "

            "Check the Kaggle download contents or update KAGGLE_DATASET if the source changes."

        )



    return max(candidates, key=lambda item: item[0])[1]





DATASET_DIR = download_asl_alphabet_dataset()

CLASS_ROOT = locate_class_root(DATASET_DIR)



print(f"Dataset directory: {DATASET_DIR}")

print(f"Class root: {CLASS_ROOT}")


Resuming download from 106954752 bytes (993932282 bytes left)...
Resuming download to C:\Users\samri\.cache\kagglehub\datasets\grassknoted\asl-alphabet\1.archive (106954752/1100887034) bytes left.


100%|██████████| 1.03G/1.03G [1:15:49<00:00, 218kB/s] 

Extracting files...


Dataset directory: C:\Users\samri\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1
Class root: C:\Users\samri\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1\asl_alphabet_train\asl_alphabet_train


In [10]:
def create_hand_landmarker(model_path=HAND_LANDMARKER_MODEL_PATH):

    resolved_model_path = ensure_hand_landmarker_model(model_path=model_path)

    options = HandLandmarkerOptions(

        base_options=BaseOptions(model_asset_path=str(resolved_model_path)),

        running_mode=RunningMode.IMAGE,

        num_hands=1,

        min_hand_detection_confidence=0.5,

        min_hand_presence_confidence=0.5,

        min_tracking_confidence=0.5,

    )

    return HandLandmarker.create_from_options(options)





def build_image_records(class_root, max_samples_per_class=None):

    records = []

    class_dirs = sorted(path for path in class_root.iterdir() if path.is_dir())



    for class_dir in class_dirs:

        image_paths = sorted(

            path

            for path in class_dir.iterdir()

            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

        )

        if max_samples_per_class is not None:

            image_paths = image_paths[:max_samples_per_class]

        for image_path in image_paths:

            records.append((image_path, class_dir.name))



    if not records:

        raise ValueError(f"No images found under {class_root}")



    return records





def normalize_landmarks(coords):

    coords = coords.astype(np.float32).copy()

    coords -= coords[0]

    scale = np.max(np.linalg.norm(coords[:, :2], axis=1))

    if scale > 0:

        coords /= scale

    return coords





def extract_landmarks_from_bgr(image_bgr, landmarker, normalize=True):

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    results = landmarker.detect(mp_image)

    if not results.hand_landmarks:

        return None



    coords = np.array(

        [[landmark.x, landmark.y, landmark.z] for landmark in results.hand_landmarks[0]],

        dtype=np.float32,

    )



    if normalize:

        coords = normalize_landmarks(coords)



    return coords.reshape(-1)





def extract_landmarks_from_image_path(image_path, landmarker, normalize=True):

    image = cv2.imread(str(image_path))

    if image is None:

        return None

    return extract_landmarks_from_bgr(image, landmarker=landmarker, normalize=normalize)





def extract_landmarks_for_inference(image_bgr, normalize=True, landmarker=None):

    if landmarker is not None:

        return extract_landmarks_from_bgr(image_bgr, landmarker=landmarker, normalize=normalize)



    with create_hand_landmarker() as local_landmarker:

        return extract_landmarks_from_bgr(image_bgr, landmarker=local_landmarker, normalize=normalize)





records = build_image_records(CLASS_ROOT, max_samples_per_class=MAX_SAMPLES_PER_CLASS)

label_counts = Counter(label for _, label in records)



print(f"Classes: {len(label_counts)}")

print(f"Images queued: {len(records)}")

print("First five labels:", list(sorted(label_counts.items()))[:5])


Classes: 29
Images queued: 1450
First five labels: [('A', 50), ('B', 50), ('C', 50), ('D', 50), ('E', 50)]


## Landmark Extraction and Splits

The next two cells save the core artifacts needed by the rest of the project: `landmarks.npy`, `labels.npy`, `splits.npz`, and `label_to_index.json`.

In [11]:
X = []

y = []

kept_paths = []

failed_paths = []



with create_hand_landmarker() as landmarker:

    for index, (image_path, label) in enumerate(records, start=1):

        landmarks = extract_landmarks_from_image_path(

            image_path,

            landmarker=landmarker,

            normalize=True,

        )

        if landmarks is None:

            failed_paths.append(str(image_path))

            continue



        X.append(landmarks)

        y.append(label)

        kept_paths.append(str(image_path))



        if index % 1000 == 0:

            print(f"Processed {index}/{len(records)} images", end="\r")



print()



X = np.asarray(X, dtype=np.float32)

y = np.asarray(y)



np.save(LANDMARKS_PATH, X)

np.save(LABELS_PATH, y)



with METADATA_PATH.open("w", newline="", encoding="utf-8") as handle:

    writer = csv.writer(handle)

    writer.writerow(["image_path", "label"])

    writer.writerows(zip(kept_paths, y.tolist()))



FAILED_IMAGES_PATH.write_text("\n".join(failed_paths), encoding="utf-8")



print(f"Saved {LANDMARKS_PATH.name} with shape {X.shape}")

print(f"Saved {LABELS_PATH.name} with shape {y.shape}")

print(f"Missing detections: {len(failed_paths)}")


Processed 1000/1450 images
Saved landmarks.npy with shape (1239, 63)
Saved labels.npy with shape (1239,)
Missing detections: 211


In [12]:
unique_labels = sorted(set(y.tolist()))
label_to_index = {label: index for index, label in enumerate(unique_labels)}
y_indexed = np.asarray([label_to_index[label] for label in y], dtype=np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_indexed,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_indexed,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

np.savez(
    SPLITS_PATH,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
)

LABEL_MAP_PATH.write_text(json.dumps(label_to_index, indent=2), encoding="utf-8")

print("Split shapes:")
print("  train:", X_train.shape, y_train.shape)
print("  val:  ", X_val.shape, y_val.shape)
print("  test: ", X_test.shape, y_test.shape)

Split shapes:
  train: (867, 63) (867,)
  val:   (186, 63) (186,)
  test:  (186, 63) (186,)


## Reusable Inference Extraction and Webcam Test Frames

Run this locally to capture a few frames from your webcam. Press the space bar to save a frame and `Q` to stop early. The cell also reuses the inference extraction helper to produce `webcam_landmarks.npy` when detections are found.

In [13]:
def extract_landmarks_from_image_file(image_path, landmarker=None):

    image = cv2.imread(str(image_path))

    if image is None:

        raise FileNotFoundError(f"Could not read image: {image_path}")

    return extract_landmarks_for_inference(image, landmarker=landmarker)





def capture_webcam_test_frames(

    output_dir=WEBCAM_OUTPUT_DIR,

    frame_count=WEBCAM_FRAME_COUNT,

    camera_index=CAMERA_INDEX,

):

    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)



    camera = cv2.VideoCapture(camera_index)

    if not camera.isOpened():

        raise RuntimeError("Could not open the webcam. Try a different CAMERA_INDEX.")



    saved_paths = []

    try:

        while len(saved_paths) < frame_count:

            ok, frame = camera.read()

            if not ok:

                raise RuntimeError("Failed to read a frame from the webcam.")



            preview = frame.copy()

            remaining = frame_count - len(saved_paths)

            cv2.putText(

                preview,

                f"SPACE capture | Q quit | remaining {remaining}",

                (10, 30),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.7,

                (0, 255, 0),

                2,

                cv2.LINE_AA,

            )

            cv2.imshow("ASL webcam capture", preview)



            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):

                break

            if key == ord(" "):

                frame_path = output_dir / f"frame_{len(saved_paths):03d}.jpg"

                cv2.imwrite(str(frame_path), frame)

                saved_paths.append(frame_path)

                print(f"Saved {frame_path.name}")

    finally:

        camera.release()

        cv2.destroyAllWindows()



    return saved_paths





captured_frames = capture_webcam_test_frames()

captured_vectors = []



with create_hand_landmarker() as landmarker:

    for frame_path in captured_frames:

        vector = extract_landmarks_from_image_file(frame_path, landmarker=landmarker)

        if vector is not None:

            captured_vectors.append(vector)



if captured_vectors:

    webcam_vectors = np.asarray(captured_vectors, dtype=np.float32)

    np.save(WEBCAM_LANDMARKS_PATH, webcam_vectors)

    print(f"Saved {WEBCAM_LANDMARKS_PATH.name} with shape {webcam_vectors.shape}")

else:

    print("No hand detections were found in the captured frames.")



print(f"Captured frames: {len(captured_frames)}")


No hand detections were found in the captured frames.
Captured frames: 0


## Expected Outputs



After running the notebook, the project folder will contain `hand_landmarker.task`, `landmarks.npy`, `labels.npy`, `splits.npz`, `label_to_index.json`, `landmark_metadata.csv`, `failed_images.txt`, `webcam_frames/`, and optionally `webcam_landmarks.npy`.
